# QuantJourney SDK - CFTC Commitment of Traders (COT)

This notebook demonstrates CFTC data:
- Commitment of Traders reports
- Commercial vs Non-Commercial positioning
- Futures market sentiment

**API:** https://api.quantjourney.cloud

## Run Output

![10_cftc_cot](../plots/10_cftc_cot_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
# PNG for GitHub, interactive in Jupyter
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "png"

from plotly.subplots import make_subplots

# API Key authentication
import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")

qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. COT Report - S&P 500 Futures

In [ ]:
# Fetch COT report for S&P 500 futures
# First, get the cftc_contract_market_code from popular symbols
popular = qj.cftc.get_popular_symbols()
popular_symbols = popular.get('value', popular) if isinstance(popular, dict) else popular
print("Available popular symbols:")
for name, code in popular_symbols.items():
    print(f"  {name}: {code}")

# Get COT data using the code
sp500_code = popular_symbols.get('SP500', '13874A')
try:
    response = qj.cftc.get_cot_data(symbol=sp500_code, report_type="legacy")
    cot_data = response.get('value', response) if isinstance(response, dict) else response
    
    if cot_data:
        df_cot = pd.DataFrame(cot_data) if isinstance(cot_data, list) else pd.DataFrame([cot_data])
        
        print(f"\nCOT records: {len(df_cot)}")
        print(f"\nKey columns:")
        key_cols = ['report_date_as_yyyy_mm_dd', 'contract_market_name', 'open_interest_all',
                   'noncomm_positions_long_all', 'noncomm_positions_short_all',
                   'comm_positions_long_all', 'comm_positions_short_all']
        for col in key_cols:
            if col in df_cot.columns:
                print(f"  - {col}")
        print(f"\nLatest data:")
        print(df_cot[['report_date_as_yyyy_mm_dd', 'open_interest_all', 
                      'comm_positions_long_all', 'comm_positions_short_all']].tail(3))
    else:
        print("No data returned")
except Exception as e:
    print(f"Note: COT data - {e}")


In [ ]:
# Process and plot COT data
if 'df_cot' in dir() and len(df_cot) > 0:
    # Convert date column
    if 'report_date_as_yyyy_mm_dd' in df_cot.columns:
        df_cot['date'] = pd.to_datetime(df_cot['report_date_as_yyyy_mm_dd'])
        df_cot = df_cot.sort_values('date').tail(104)  # Last 2 years
        
        fig = go.Figure()
        
        # Commercial positions
        if 'comm_positions_long_all' in df_cot.columns:
            fig.add_trace(go.Scatter(
                x=df_cot['date'],
                y=df_cot['comm_positions_long_all'],
                name='Commercial Long',
                line=dict(color='green')
            ))
        
        if 'comm_positions_short_all' in df_cot.columns:
            fig.add_trace(go.Scatter(
                x=df_cot['date'],
                y=df_cot['comm_positions_short_all'],
                name='Commercial Short',
                line=dict(color='red')
            ))
        
        # Non-commercial positions
        if 'noncomm_positions_long_all' in df_cot.columns:
            fig.add_trace(go.Scatter(
                x=df_cot['date'],
                y=df_cot['noncomm_positions_long_all'],
                name='Non-Commercial Long',
                line=dict(color='blue', dash='dash')
            ))
        
        if 'noncomm_positions_short_all' in df_cot.columns:
            fig.add_trace(go.Scatter(
                x=df_cot['date'],
                y=df_cot['noncomm_positions_short_all'],
                name='Non-Commercial Short',
                line=dict(color='orange', dash='dash')
            ))
        
        fig.update_layout(
            title='COT - S&P 500 E-Mini Futures Positioning',
            yaxis_title='Contracts',
            xaxis_title='Date',
            template='plotly_dark',
            height=500
        )
        fig.show()
        
        # Calculate net positioning
        if all(col in df_cot.columns for col in ['comm_positions_long_all', 'comm_positions_short_all']):
            df_cot['comm_net'] = df_cot['comm_positions_long_all'] - df_cot['comm_positions_short_all']
            df_cot['noncomm_net'] = df_cot['noncomm_positions_long_all'] - df_cot['noncomm_positions_short_all']
            
            print(f"\nLatest Positioning Summary:")
            latest = df_cot.iloc[-1]
            print(f"  Date: {latest['date'].strftime('%Y-%m-%d')}")
            print(f"  Open Interest: {latest['open_interest_all']:,.0f}")
            print(f"  Commercial Net: {latest['comm_net']:,.0f}")
            print(f"  Non-Commercial Net: {latest['noncomm_net']:,.0f}")
else:
    print("No COT data available")


## 2. COT Report - Gold Futures

In [ ]:
# Fetch COT report for Gold futures
gold_code = popular_symbols.get('GOLD', '088691')
try:
    response = qj.cftc.get_cot_data(symbol=gold_code, report_type="legacy")
    gold_cot = response.get('value', response) if isinstance(response, dict) else response
    
    if gold_cot:
        df_gold_cot = pd.DataFrame(gold_cot) if isinstance(gold_cot, list) else pd.DataFrame([gold_cot])
        
        if 'report_date_as_yyyy_mm_dd' in df_gold_cot.columns:
            df_gold_cot['date'] = pd.to_datetime(df_gold_cot['report_date_as_yyyy_mm_dd'])
            df_gold_cot = df_gold_cot.sort_values('date')
            
            print(f"Gold COT records: {len(df_gold_cot)}")
            print(f"\nLatest data:")
            print(df_gold_cot[['report_date_as_yyyy_mm_dd', 'open_interest_all', 
                              'comm_positions_long_all', 'comm_positions_short_all']].tail(3))
    else:
        print("No Gold COT data returned")
except Exception as e:
    print(f"Note: Gold COT data - {e}")


## 3. COT Interpretation

In [ ]:
# COT analysis summary
print("="*60)
print("COMMITMENT OF TRADERS (COT) ANALYSIS")
print("="*60)

print("\nCOT REPORT CATEGORIES:")
categories = [
    ("Commercial", "Hedgers - producers/consumers of the commodity"),
    ("Non-Commercial", "Large speculators - hedge funds, CTAs"),
    ("Non-Reportable", "Small speculators - retail traders"),
]

for name, desc in categories:
    print(f"   • {name:18} - {desc}")

print("\nINTERPRETATION:")
signals = [
    "Extreme commercial long = potential bottom (smart money buying)",
    "Extreme commercial short = potential top (smart money selling)",
    "Non-commercial extreme long = potential top (crowded trade)",
    "Non-commercial extreme short = potential bottom (panic selling)",
]

for signal in signals:
    print(f"   • {signal}")

print("\n" + "="*60)


## Summary

CFTC COT data covered:
- **Commitment of Traders**: Weekly positioning reports
- **Commercial Positions**: Hedger activity
- **Non-Commercial Positions**: Speculator activity

### Key Insights:
- Commercial traders are typically "smart money"
- Extreme positioning can signal reversals
- Net positioning changes show sentiment shifts